# SimpleLLM V0.32 — PyTorch (Stability Patch)

**Fixes in this version:**
- **CUDA OOM Fix:** Reduced batch size to 32 and moved EVAL/GEN after Backward to avoid memory spikes.
- **Tokenizer Fix:** Set `model_max_length` to suppress sequence length warnings.
- **Memory Optimization:** Added `torch.cuda.empty_cache()` after heavy eval blocks.

Updated: 2026-02-27
**Dataset:** Now dataset contatins only wiki text.

In [ ]:
# ========================== [CELL 1] DEPENDENCIES ==========================
# This block imports all required libraries for the project.
# I tried to use only the most basic PyTorch dependencies to
# demonstrate an understanding of neural networks mechanics "under the hood".

import math           # For mathematical operations (e.g. RoPE and LR Scheduler)
import os             # For file system interactions (saving checkpoints)
import time           # To measure training time and output statistics
from typing import Optional, Tuple

import numpy as np    # Library for working with arrays
import torch          # The core deep learning framework
import torch.nn as nn # Module containing neural network layers
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

# Automatically select GPU (CUDA) if available, otherwise fallback to CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

In [ ]:
# ========================== [CELL 2] HYPERPARAMETERS ==========================
# This cell defines the neural network's architecture. These parameters determine its
# "brain power". The model is built on LLaMA principles but scaled down
# to fit the capabilities of a single graphics card.

class ModelArgs:
    dim: int = 512          # Hidden layer dimension (embeddings size)
    n_layers: int = 6       # Number of transformer layers
    n_heads: int = 8        # Number of attention heads (Multi-Head Attention)
    vocab_size: int = -1    # Vocabulary size (will be calculated from the tokenizer)
    multiple_of: int = 256  # Base dimension for SwiGLU activation function (taken from LLaMA)
    norm_eps: float = 1e-5  # Epsilon for RMSNorm (to prevent division by zero)
    max_seq_len: int = 256  # Maximum context length. Limited to prevent memory issues (OOM)
    dropout: float = 0.1    # Probability of dropping a neuron, acts as a defense against overfitting

class TrainArgs:
    batch_size: int = 32         # Batch size, reduced to 32 for OOM-safety -> this fixed the memory issue
    learning_rate: float = 5e-4  # Initial learning rate
    min_lr: float = 1e-5         # Minimum learning rate (reached at the end of the cosine decay)
    warmup_steps: int = 1000     # Warmup steps to prevent weights from "exploding" early in training
    max_steps: int = 20000       # Total number of iterations
    weight_decay: float = 0.05   # Regularization (L2 penalty) to reduce overfitting
    grad_clip: float = 1.0       # Gradient clipping for stability (prevents exploding gradients)
    eval_steps: int = 500        # How often we generate text to check progress

config = ModelArgs()
train_config = TrainArgs()

In [ ]:
# ========================== [CELL 3] TOKENIZER & DATASET ==========================
# This section handles data preprocessing for training the language model.
# To us, text is just a string of letters, but for a neural network, we must translate it
# into numbers — "tokens". The Tokenizer class performs this process.

from transformers import AutoTokenizer
from datasets import load_dataset
import torch
from torch.utils.data import IterableDataset, DataLoader

# We use the GPT-2 tokenizer. It is reliable and has a rich vocabulary (about 50k words/subwords).
# Bug fix: We suppress context length warnings by intentionally setting a huge model_max_length.
tokenizer = AutoTokenizer.from_pretrained("gpt2", model_max_length=1e9)
tokenizer.pad_token = tokenizer.eos_token  # Treat the "padding" token as "end of sentence".
config.vocab_size = tokenizer.vocab_size   # Automatically write the vocabulary size to the config.

# Load the Wikitext-103 dataset. The dataset contains a variety of topics and high-quality
# "adult" grammar from Wikipedia, which makes the model "smarter" in the early stages.
# streaming=True means "don't load into RAM all at once, read in chunks" - this saves computer's memory.
dataset = load_dataset("wikitext", "wikitext-103-v1", split="train", streaming=True)
val_dataset = load_dataset("wikitext", "wikitext-103-v1", split="validation", streaming=True)

# This class handles packing tokens. It reads text, translates it into numerical tokens,
# and outputs chunks of length (max_seq_len + 1) to the model. It does this "on the fly" (Iterable).
class PackedTokenizedDataset(IterableDataset):
    def __init__(self, dataset, tokenizer, max_seq_len):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        
    def __iter__(self):
        buffer = []
        for sample in self.dataset:
            if not sample.get('text', '').strip(): continue
            tokens = self.tokenizer.encode(sample['text'], add_special_tokens=False)
            tokens.append(self.tokenizer.eos_token_id)
            buffer.extend(tokens)
            # Yield elements as long as the buffer contains more items than the window length
            while len(buffer) >= self.max_seq_len + 1:
                chunk = buffer[:self.max_seq_len + 1]
                buffer = buffer[self.max_seq_len:]
                # Feed x (input) and y (target - shifted by 1 character) to the model
                yield torch.tensor(chunk[:-1], dtype=torch.long), torch.tensor(chunk[1:], dtype=torch.long)

train_dataloader = DataLoader(PackedTokenizedDataset(dataset, tokenizer, config.max_seq_len), batch_size=train_config.batch_size)
val_dataloader = DataLoader(PackedTokenizedDataset(val_dataset, tokenizer, config.max_seq_len), batch_size=train_config.batch_size)

In [ ]:
# ========================== [CELL 4] MODULES ==========================
# This block contains 3 crucial components of modern Language Models.
# Most of them were pioneered by Meta's LLaMA model!

# 1. RMSNorm (Root Mean Square Normalization). A simpler and faster version of
# standard LayerNorm. It does not require calculating the mean, only dividing
# by the root of the squares. The goal is to stabilize training!
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim)) # Learnable weight
    def forward(self, x): 
        # rsqrt is 1 / sqrt(x^2 + e). "pow(2)" squares the elements.
        return self.weight * (x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps))

# 2. Rotary Position Embeddings (RoPE). This is how the network understands "where a word is located".
# Unlike older networks (where an index was simply added), here the token's position
# is encoded as a complex rotation (sines/cosines) in the hidden space of
# Key and Query. We compute the matrix of their angles here beforehand (precompute):
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    freqs = torch.outer(torch.arange(end), freqs).float()
    return torch.polar(torch.ones_like(freqs), freqs)

# This function simply multiplies Q and K by the "rotation" matrix
def apply_rotary_emb(xq, xk, freqs_cis):
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.view(1, xq_.shape[1], 1, xq_.shape[-1])
    return torch.view_as_real(xq_ * freqs_cis).flatten(3).type_as(xq), torch.view_as_real(xk_ * freqs_cis).flatten(3).type_as(xk)

# 3. SwiGLU. An incredibly efficient type of "Feed-Forward" network (FC layer).
# Instead of good old ReLU, this uses SiLU (or Swish) multiplied by a
# linear layer (Gated Linear Unit). This is the modern industry standard.
class SwiGLU(nn.Module):
    def __init__(self, dim: int, multiple_of: int):
        super().__init__()
        # Tricky alignment of the layer's dimension to ensure hardware efficiency (multiple_of -> 256)
        hidden_dim = multiple_of * ((int(2 * 2 * dim / 3) + multiple_of - 1) // multiple_of)
        self.w1, self.w2, self.w3 = nn.Linear(dim, hidden_dim, bias=False), nn.Linear(hidden_dim, dim, bias=False), nn.Linear(dim, hidden_dim, bias=False)
        
    def forward(self, x): 
        # SiLU (x * sigmoid(x)) is multiplied by the output of layer W3. A "Gate" mechanism.
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

In [ ]:
# ========================== [CELL 5] ATTENTION ==========================
# Attention is the "core" of Transformers! It's a mechanism that looks at "which words
# are more important for understanding the current one." The essence lies in a Query,
# a Key, and a Value. Q is compared with K to compute the "attention strength",
# and then this weight is multiplied by V.

class Attention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        # The model is split into "heads". This allows for studying various aspects of a word:
        # The 1st head looks at grammar, the 2nd at noun relations, etc.
        self.n_heads, self.head_dim = args.n_heads, args.dim // args.n_heads
        
        # Linear projections without a bias, like in LLaMA:
        self.wq, self.wk, self.wv, self.wo = [nn.Linear(args.dim, args.dim, bias=False) for _ in range(3)] + [nn.Linear(args.dim, args.dim, bias=False)]
        
        self.resid_dropout = nn.Dropout(args.dropout) # To prevent over-learning on the training set

    def forward(self, x, freqs_cis, use_cache=False, kv_cache=None):
        bsz, seqlen, _ = x.shape
        # Prepare Q, K, V
        xq, xk, xv = self.wq(x).view(bsz, seqlen, self.n_heads, self.head_dim), self.wk(x).view(bsz, seqlen, self.n_heads, self.head_dim), self.wv(x).view(bsz, seqlen, self.n_heads, self.head_dim)
        
        # Encode positions (RoPE)
        xq, xk = apply_rotary_emb(xq, xk, freqs_cis)
        
        # KV Cache mechanism. Only active during text generation!
        # The network doesn't have to recalculate the past on each step; it simply remembers it.
        if use_cache:
            if kv_cache: xk, xv = torch.cat([kv_cache[0], xk], 1), torch.cat([kv_cache[1], xv], 1)
            new_cache = (xk, xv)
        else: new_cache = None
            
        # Using the highly optimized "Flash Attention" equivalent: `scaled_dot_product_attention`.
        # The causal mask (seqlen > 1) cuts off "peeking into the future", forcing the network to guess the next word.
        out = F.scaled_dot_product_attention(xq.transpose(1, 2), xk.transpose(1, 2), xv.transpose(1, 2), is_causal=(seqlen > 1))
        
        # Merge the head matrices back together and apply dropout
        return self.resid_dropout(self.wo(out.transpose(1, 2).contiguous().view(bsz, seqlen, -1))), new_cache

In [ ]:
# ========================== [CELL 6] MODEL ==========================
# Describing the complete components of a Transformer Block and the network itself.

class TransformerBlock(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        # Each block contains Attention and Feed-Forward (SwiGLU) wrappers
        self.attention, self.feed_forward = Attention(args), SwiGLU(args.dim, args.multiple_of)
        # We also put normalization layers before them
        self.attention_norm, self.ffn_norm = RMSNorm(args.dim, eps=args.norm_eps), RMSNorm(args.dim, eps=args.norm_eps)
        
    def forward(self, x, freqs_cis, use_cache=False, kv_cache=None):
        # Residual Connections (x + h). We add the block's output back to its input,
        # so we don't "lose the original meaning", and it helps avoid vanishing gradients during training!
        h, new_cache = self.attention(self.attention_norm(x), freqs_cis, use_cache, kv_cache)
        return x + h + self.feed_forward(self.ffn_norm(x + h)), new_cache

# Assembling the entire "machine"
class ModernLLM(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        # Embeddings translate vocabulary (token indices) -> into vectors (dim=512 default)
        self.tok_embeddings = nn.Embedding(args.vocab_size, args.dim)
        
        # Build N transformer layers
        self.layers = nn.ModuleList([TransformerBlock(args) for _ in range(args.n_layers)])
        
        # Final layer normalization and output layer.
        self.norm, self.output = RMSNorm(args.dim, eps=args.norm_eps), nn.Linear(args.dim, args.vocab_size, bias=False)
        
        # Weight Tying: an extremely cool trick where we force the output matrix
        # to use the same weights as the embedding matrix. It saves memory (parameters) by ~20-30%!
        self.tok_embeddings.weight = self.output.weight
        
        # Memory Buffer: Precomputed rotation angles for RoPE; they do not need to be "trained".
        self.register_buffer("freqs_cis", precompute_freqs_cis(args.dim // args.n_heads, args.max_seq_len * 2))
        
        # Apply custom initialization to start off with a good "weight state"
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, tokens, use_cache=False, kv_caches=None):
        h = self.tok_embeddings(tokens)
        
        # If we're using cache, we adjust what portion of the RoPE matrix (positions) to use
        start_pos = 0 if not kv_caches else kv_caches[0][0].shape[1]
        freqs_cis = self.freqs_cis[start_pos : start_pos + tokens.shape[1]]
        
        new_caches = []
        for i, layer in enumerate(self.layers):
            h, nc = layer(h, freqs_cis, use_cache, kv_caches[i] if kv_caches else None)
            if use_cache: new_caches.append(nc)
            
        # Logits are the "probabilities" of which word comes next.
        logits = self.output(self.norm(h))
        return (logits, new_caches) if use_cache else logits

# Instantiate the structure and send it to the GPU (or CPU)
model = ModernLLM(config).to(device)

In [ ]:
# ========================== [CELL 7] GENERATION ==========================
# Let's try to make this thing speak. The "generation" function evaluates
# tokens one by one in a loop until it encounters a special
# "EOS" (End of Sentence) token. Done under the no_grad decorator for speed - during
# generation we do not need gradients (training).

@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=100, temperature=0.8, top_p=0.9):
    model.eval()  # Switch to "evaluation" mode
    tokens = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    # 1. First initial pass of the user prompt with KV Caching
    logits, kv_caches = model(tokens, use_cache=True)
    curr_logit = logits[:, -1, :]
    
    # Fix: use tokens[0] instead of squeeze() so single-word prompts stay as a list,
    # preventing the "'int' object has no attribute 'append'" error.
    res = tokens[0].tolist()
    
    for _ in range(max_new_tokens):
        # Temperature smooths or "sharpens" the probability distribution (so the AI isn't boring)
        p = torch.softmax(curr_logit / (temperature + 1e-10), -1)
        
        # 2. Top-P (Nucleus) Sampling: we take options whose cumulative probability makes up 90%
        # The rest are dropped (protects against pure gibberish generation).
        sorted_p, sorted_i = torch.sort(p, descending=True)
        cp = torch.cumsum(sorted_p, -1)
        
        # Zero out the ones out of bounds
        mask = cp - sorted_p > top_p
        sorted_p[mask] = 0.0
        sorted_p = sorted_p / sorted_p.sum(dim=-1, keepdim=True) # Bring the sum back to 100%
        
        # 3. Sample the next token and record it
        next_token_idx = torch.multinomial(sorted_p, 1)
        nt = torch.gather(sorted_i, -1, next_token_idx).item()
        
        res.append(nt)
        if nt == tokenizer.eos_token_id: break
            
        # 4. Pass the NEXT token, using the CACHE! The model does not have to reread everything from scratch
        logits, kv_caches = model(torch.tensor([[nt]], device=device), use_cache=True, kv_caches=kv_caches)
        curr_logit = logits[:, -1, :]
        
    return tokenizer.decode(res)

In [ ]:
# ========================== [CELL 8] TRAINING LOOP ==========================
# And finally, the training loop! This is the part where our model "learns" by penalizing errors.
import time
import math
import os

# Auto-detect directory for Kaggle or local execution
output_dir = "/kaggle/working/" if os.path.exists("/kaggle/working/") else "./"

# AdamW Optimizer - industry standard. Takes advantage of momentum to stabilize learning.
optimizer = torch.optim.AdamW(model.parameters(), lr=train_config.learning_rate, betas=(0.9, 0.95), weight_decay=train_config.weight_decay)

# Learning Rate Scheduler:
# We first perform linear warmup (so we don't blow up weights at the start),
# and then a smooth cosine decay down to 0
def get_lr_multiplier(step):
    if step < train_config.warmup_steps:
        return step / max(1, train_config.warmup_steps) # Linear scaling to peak
    progress = (step - train_config.warmup_steps) / max(1, train_config.max_steps - train_config.warmup_steps)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress))) # Cosine decay to 0

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr_multiplier)
# Gradient Scaler for Mixed Precision (FP16), allowing to train ~30% faster and prevent memory exhaustion.
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

train_iter = iter(train_dataloader)
model.train() # Enable Dropout

print(f"Starting training for {train_config.max_steps} steps! (Batch_size={train_config.batch_size})...")
print(f"Models will be saved to: {output_dir}")
start_time = time.time()
running_loss = 0.0

for step in range(train_config.max_steps):
    step_start = time.time()
    
    # 1. Take a new piece of data
    try:
        X, Y = next(train_iter)
    except StopIteration:
        # If the data runs out (rare with streaming datasets), we restart the iterator
        train_iter = iter(train_dataloader)
        X, Y = next(train_iter)
        
    X, Y = X.to(device), Y.to(device)
    optimizer.zero_grad(set_to_none=True) # Clean up old gradient errors from the previous step
    
    # Automatic Mixed Precision: computes heavy layers in FP16 (float16) rather than FP32 (float32).
    with torch.amp.autocast(device.type, dtype=torch.float16):
        # Forward pass. Instantly computes CrossEntropy (comparing prediction to ground truth)
        loss = F.cross_entropy(model(X).view(-1, config.vocab_size), Y.view(-1))
    
    # Backward pass - backpropagating error through the graphs.
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    
    # "Gradient clipping" (Protection against NaN losses)
    torch.nn.utils.clip_grad_norm_(model.parameters(), train_config.grad_clip)
    
    # Perform optimization step!
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    # Moving average for smooth logging (so the plot doesn't jump rapidly)
    running_loss = loss.item() if step == 0 else running_loss * 0.99 + loss.item() * 0.01

    # --- LOGGING ---
    if step % 50 == 0:
        step_time = time.time() - step_start
        elapsed = time.time() - start_time
        steps_left = train_config.max_steps - step
        eta_seconds = steps_left * (elapsed / max(1, step)) if step > 0 else 0
        
        current_lr = optimizer.param_groups[0]['lr']
        # Perplexity (PPL) - key metric of how well the model predicts language.
        # Perfect value is 1. The lower, the better.
        train_ppl = math.exp(running_loss) if running_loss < 20 else float('inf')
        
        print(f"Step {step:5d}/{train_config.max_steps} | "
              f"Loss: {loss.item():.4f} (Avg: {running_loss:.4f}) | "
              f"PPL: {train_ppl:.2f} | "
              f"LR: {current_lr:.2e} | "
              f"Time/step: {step_time*1000:.1f}ms | "
              f"ETA: {eta_seconds/60:.1f}m")

    # --- EVALUATION AND GENERATION ---
    if step > 0 and step % train_config.eval_steps == 0:
        model.eval()  # Very important to turn off Dropout during evaluation
        print(f"\n{'-'*60}\n=== EVALUATION AT STEP {step} ===")
        
        val_losses = []
        val_iter = iter(val_dataloader)
        with torch.no_grad():
            for _ in range(10): # Limit eval to 10 batches
                try:
                    vX, vY = next(val_iter)
                except StopIteration:
                    break
                vX, vY = vX.to(device), vY.to(device)
                with torch.amp.autocast(device.type, dtype=torch.float16):
                    v_loss = F.cross_entropy(model(vX).view(-1, config.vocab_size), vY.view(-1))
                val_losses.append(v_loss.item())
        
        avg_val_loss = sum(val_losses) / len(val_losses) if val_losses else 0.0
        val_ppl = math.exp(avg_val_loss) if avg_val_loss < 20 else float('inf')
        
        print(f"Validation Loss: {avg_val_loss:.4f} | Validation PPL: {val_ppl:.2f}")
        
        # Checking generation visually - the most exciting and visual moment!
        print(f"Sample Generation:")
        sample_text = generate(model, tokenizer, 'Once upon a time', max_new_tokens=40)
        print(f"> {sample_text}\n{'-'*60}\n")
        
        model.train() # Return to training mode
        torch.cuda.empty_cache() # Fix RAM leak (Memory Optimization!)
        
    # --- CHECKPOINT SAVING ---
    if step > 0 and step % 5000 == 0:
        checkpoint_path = os.path.join(output_dir, f"model_step_{step}.pt")
        torch.save(model.state_dict(), checkpoint_path)
        print(f"[SUCCESS] Checkpoint saved: {checkpoint_path}")

# --- FINALS ---
final_model_path = os.path.join(output_dir, "SimpleLLM_V032_Final.pt")
torch.save(model.state_dict(), final_model_path)
print(f"\nTraining Complete! Final model ready at: {final_model_path}")

In [ ]:
# ========================== [CELL 9] LOAD MODEL & INTERACTIVE TESTING ==========================
# Full setup for Interactive chatting with our freshly trained model!
import os

output_dir = "/kaggle/working/" if os.path.exists("/kaggle/working/") else "./"

# Restoring savings: you can restart the kernel and just load this file,
# so you don't have to train for 2 hours again before the presentation.
model_path = os.path.join(output_dir, "SimpleLLM_V032_Final.pt")
if os.path.exists(model_path):
    print(f"Attempting to load weights from: {model_path}...")
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print("Success! Model loaded!")
else:
    print("No saved models found. Using the model currently living in memory (just trained).")

print("\n=== INTERACTIVE TESTING ===")
print("Enter any phrase, and the model will continue it. Type 'quit' or 'exit' to stop.")

while True:
    prompt = input("\nYour Prompt: ")
    if prompt.lower() in ['quit', 'exit']:
        print("Testing complete.")
        break
    
    if not prompt.strip():
        continue
        
    print("Analyzing and generating response...\n")
    # Temperature 0.8 chosen for creative responses, max_new_tokens=100 for proper length
    output = generate(model, tokenizer, prompt, max_new_tokens=100, temperature=0.8, top_p=0.9)
    print(f"[Model Output]:\n{output}\n")
    print("-" * 50)